In [11]:
import pandas as pd

In [12]:
slantlabel = pd.read_csv("../news_article_slant_classification/data/news_articles_leaning.csv")
slantlabel.slantlabel_final.value_counts()

slantlabel_final
4.0    1502
3.5     777
3.0     558
4.5     380
2.5     335
5.0     203
2.0     186
6.5     140
5.5     139
6.0     118
1.5      65
7.0      27
1.0       4
Name: count, dtype: int64

In [13]:
base_dir = "/data1/shared_datasets/project-training-data/olmo3"
dataset2file = {"RefinedWeb": f"{base_dir}/refinedweb/refinedweb_classified_left_right.csv",
                "Pile": f"{base_dir}/pile/pile_classified_left_right.csv",
                "Dolma3": f"{base_dir}/dolma3/dolma3_left_right.csv",
                "Dolmino3": f"{base_dir}/dolmino3/dolmino_classified_left_right.csv",
                "SFT3": f"{base_dir}/sft/sft_left_right.csv",
                "DPO3": f"{base_dir}/dpo/dpo_left_right.csv"}

In [14]:
import tldextract

_extract = tldextract.TLDExtract(suffix_list_urls=())  # offline, bundled suffix list

def domain_root(url):
    """'https://www.cnn.com/politics/x' -> 'cnn.com'"""
    if pd.isna(url):
        return None
    e = _extract(str(url))
    return f"{e.domain}.{e.suffix}" if e.domain and e.suffix else None

# article-level label (same slant -> class map as leaning_classification/process_data.py)
slantlabel["domain"] = slantlabel["canonicalurl"].map(domain_root)
slantlabel["label"] = slantlabel["slantlabel_final"].map(
    lambda x: "left" if x <= 3.5 else ("right" if x >= 4.5 else "neutral")
)

# outlet-level leaning: left if more of its articles are left than right, right otherwise
outlet_counts = slantlabel.groupby(["domain", "label"]).size().unstack(fill_value=0)
outlet_counts = outlet_counts.reindex(columns=["left", "right", "neutral"], fill_value=0)
outlet_counts["leaning"] = "tie"
outlet_counts.loc[outlet_counts.left > outlet_counts.right, "leaning"] = "left"
outlet_counts.loc[outlet_counts.right > outlet_counts.left, "leaning"] = "right"
print(len(outlet_counts), "outlets")
print(outlet_counts["leaning"].value_counts())
outlet2leaning = outlet_counts["leaning"]

94 outlets
leaning
left     74
right    20
Name: count, dtype: int64


In [15]:
rows = []
for dataset, file in dataset2file.items():
    neutral_file = file.replace("left_right", "neutral")
    # RefinedWeb's neutral file stores the URL in "id" (it has no "url" column)
    neutral_col = "id" if dataset == "RefinedWeb" else "url"
    try:
        df = pd.concat([
            pd.read_csv(file, usecols=["url"]),
            pd.read_csv(neutral_file, usecols=[neutral_col]).rename(columns={neutral_col: "url"}),
        ], axis=0)
    except ValueError:  # SFT3 / DPO3 have no URL column
        print(f"{dataset}: no url column, skipped")
        continue

    leaning = df["url"].map(domain_root).map(outlet2leaning)
    counts = leaning.value_counts()
    rows.append({
        "dataset": dataset,
        "n_docs": len(df),
        "n_with_url": int(df["url"].notna().sum()),
        "left": counts.get("left", 0),
        "right": counts.get("right", 0),
    })

shares = pd.DataFrame(rows).set_index("dataset")
shares["left_%"] = 100 * shares["left"] / shares["n_docs"]
shares["right_%"] = 100 * shares["right"] / shares["n_docs"]
shares["left_%_of_left+right"] = 100 * shares["left"] / (shares["left"] + shares["right"])
shares["right_%_of_left+right"] = 100 * shares["right"] / (shares["left"] + shares["right"])
shares.round(3)

Pile: no url column, skipped
SFT3: no url column, skipped
DPO3: no url column, skipped


,n_docs,n_with_url,left,right,left_%,right_%,left_%_of_left+right,right_%_of_left+right
dataset,,,,,,,,
RefinedWeb,194335,194335,2604,426,1.340,0.219,85.941,14.059
Dolma3,473837,473837,5177,942,1.093,0.199,84.605,15.395
Dolmino3,188107,103161,183,46,0.097,0.024,79.913,20.087


In [19]:
# same analysis restricted to the left_right sample (the neutral files are not read)
rows = []
for dataset, file in dataset2file.items():
    try:
        df = pd.read_csv(file, usecols=["url"])
    except ValueError:  # SFT3 / DPO3 have no URL column
        print(f"{dataset}: no url column, skipped")
        continue

    leaning = df["url"].map(domain_root).map(outlet2leaning)
    counts = leaning.value_counts()
    rows.append({
        "dataset": dataset,
        "n_left_right_docs": len(df),
        "n_with_url": int(df["url"].notna().sum()),
        "left": counts.get("left", 0),
        "right": counts.get("right", 0),
    })

shares_lr = pd.DataFrame(rows).set_index("dataset")
shares_lr["left_%"] = 100 * shares_lr["left"] / shares_lr["n_left_right_docs"]
shares_lr["right_%"] = 100 * shares_lr["right"] / shares_lr["n_left_right_docs"]
shares_lr["left_%_of_left+right"] = 100 * shares_lr["left"] / (shares_lr["left"] + shares_lr["right"])
shares_lr["right_%_of_left+right"] = 100 * shares_lr["right"] / (shares_lr["left"] + shares_lr["right"])
shares_lr.round(3)

Pile: no url column, skipped
SFT3: no url column, skipped
DPO3: no url column, skipped


,n_left_right_docs,n_with_url,left,right,left_%,right_%,left_%_of_left+right,right_%_of_left+right
dataset,,,,,,,,
RefinedWeb,8765,8765,222,114,2.533,1.301,66.071,33.929
Dolma3,28456,28456,764,276,2.685,0.970,73.462,26.538
Dolmino3,9230,3582,31,17,0.336,0.184,64.583,35.417


In [20]:
# Bootstrap CIs for the left / right shares in cell 4 (left_right + neutral files).
# Documents from the same outlet are not independent, so we resample *domains* (cluster bootstrap)
# rather than individual documents; docs without a URL count as their own cluster.
# One-sided test of H1: left share > right share.
import numpy as np

B = 1000

def ci(x):
    return np.percentile(x, [2.5, 97.5])

def cluster_bootstrap(urls, dataset, rng):
    """Bootstrap the left/right shares of a Series of document URLs, resampling domains."""
    domain = urls.map(domain_root).to_numpy(dtype=object)
    missing = pd.isna(domain)
    domain[missing] = [f"__missing_{i}" for i in np.flatnonzero(missing)]
    leaning = pd.Series(domain).map(outlet2leaning)

    # one row per domain: number of docs, of left docs, of right docs
    per_domain = pd.DataFrame({
        "domain": domain,
        "left": (leaning == "left").to_numpy(),
        "right": (leaning == "right").to_numpy(),
    }).groupby("domain").agg(n=("left", "size"), left=("left", "sum"), right=("right", "sum"))
    n_d, l_d, r_d = per_domain.to_numpy().T
    D = len(per_domain)

    left_pct, right_pct, diff, left_share = (np.empty(B) for _ in range(4))
    for b in range(B):
        w = np.bincount(rng.integers(0, D, D), minlength=D)  # how often each domain is drawn
        N, L, R = w @ n_d, w @ l_d, w @ r_d
        left_pct[b], right_pct[b] = 100 * L / N, 100 * R / N
        diff[b] = left_pct[b] - right_pct[b]
        left_share[b] = 100 * L / (L + R)

    n, L, R = n_d.sum(), l_d.sum(), r_d.sum()
    return {
        "dataset": dataset,
        "n_docs": n,
        "n_domains": D,
        "left_%": 100 * L / n,
        "left_% CI": tuple(ci(left_pct).round(3)),
        "right_%": 100 * R / n,
        "right_% CI": tuple(ci(right_pct).round(3)),
        "left-right (pp)": 100 * (L - R) / n,
        "left-right CI": tuple(ci(diff).round(3)),
        "left-right 1-sided lower": np.percentile(diff, 5),  # 95% one-sided lower bound
        "left_%_of_left+right": 100 * L / (L + R),
        "left_share CI": tuple(ci(left_share).round(2)),
        # one-sided bootstrap p-value for H1: left > right (share of resamples with left <= right)
        "p(left>right)": (np.sum(diff <= 0) + 1) / (B + 1),
    }

def finish(rows):
    out = pd.DataFrame(rows).set_index("dataset")
    out["significant"] = out["p(left>right)"] < 0.05
    return out.round(3)

rng = np.random.default_rng(0)
rows = []
for dataset, file in dataset2file.items():
    neutral_file = file.replace("left_right", "neutral")
    # RefinedWeb's neutral file stores the URL in "id" (it has no "url" column)
    neutral_col = "id" if dataset == "RefinedWeb" else "url"
    try:
        urls = pd.concat([
            pd.read_csv(file, usecols=["url"])["url"],
            pd.read_csv(neutral_file, usecols=[neutral_col])[neutral_col].rename("url"),
        ], ignore_index=True)
    except ValueError:  # Pile / SFT3 / DPO3 have no URL column
        print(f"{dataset}: no url column, skipped")
        continue
    rows.append(cluster_bootstrap(urls, dataset, rng))

boot = finish(rows)
boot

Pile: no url column, skipped
SFT3: no url column, skipped
DPO3: no url column, skipped


,n_docs,n_domains,left_%,left_% CI,right_%,right_% CI,left-right (pp),left-right CI,left-right 1-sided lower,left_%_of_left+right,left_share CI,p(left>right),significant
dataset,,,,,,,,,,,,,
RefinedWeb,194335,120504,1.340,"(0.952, 1.804)",0.219,"(0.094, 0.384)",1.121,"(0.711, 1.583)",0.742,85.941,"(76.21, 93.45)",0.001,True
Dolma3,473837,235563,1.093,"(0.704, 1.544)",0.199,"(0.059, 0.414)",0.894,"(0.461, 1.397)",0.524,84.605,"(71.07, 95.12)",0.001,True
Dolmino3,188107,135328,0.097,"(0.061, 0.143)",0.024,"(0.005, 0.051)",0.073,"(0.026, 0.122)",0.033,79.913,"(61.53, 95.56)",0.001,True


In [22]:
def cell(x, ci, d):
    return f"{x:.{d}f} [{ci[0]:.{d}f}, {ci[1]:.{d}f}]"

rows = [
    f"{name} & {cell(r['left_%'], r['left_% CI'], 3)} & {cell(r['right_%'], r['right_% CI'], 3)} "
    f"& {cell(r['left-right (pp)'], r['left-right CI'], 3)} "
    f"& {cell(r['left_%_of_left+right'], r['left_share CI'], 1)} \\\\"
    for name, r in boot.iterrows()
]

print("\n".join([
    r"\begin{table}[t]",
    r"\centering",
    r"\small",
    r"\begin{tabular}{lllll}",
    r"\toprule",
    r"Dataset & Left (\%) & Right (\%) & Left $-$ Right (pp) & Left share of L+R (\%) \\",
    r"\midrule",
    *rows,
    r"\bottomrule",
    r"\end{tabular}",
    r"\caption{Share of documents from left- and right-leaning news outlets over the entire sample "
    r"(left, right and neutral documents), with 95\% cluster-bootstrap confidence intervals "
    r"(1000 resamples of domains). The left--right difference is significant in every dataset.}",
    r"\label{tab:outlet-leaning-shares}",
    r"\end{table}",
]))

\begin{table}[t]
\centering
\small
\begin{tabular}{lllll}
\toprule
Dataset & Left (\%) & Right (\%) & Left $-$ Right (pp) & Left share of L+R (\%) \\
\midrule
RefinedWeb & 1.340 [0.952, 1.804] & 0.219 [0.094, 0.384] & 1.121 [0.711, 1.583] & 85.9 [76.2, 93.5] \\
Dolma3 & 1.093 [0.704, 1.544] & 0.199 [0.059, 0.414] & 0.894 [0.461, 1.397] & 84.6 [71.1, 95.1] \\
Dolmino3 & 0.097 [0.061, 0.143] & 0.024 [0.005, 0.051] & 0.073 [0.026, 0.122] & 79.9 [61.5, 95.6] \\
\bottomrule
\end{tabular}
\caption{Share of documents from left- and right-leaning news outlets over the entire sample (left, right and neutral documents), with 95\% cluster-bootstrap confidence intervals (1000 resamples of domains). The left--right difference is significant in every dataset.}
\label{tab:outlet-leaning-shares}
\end{table}
